# Domain Q&A RAG Assistant Demo

This notebook demonstrates the core functionality of the RAG pipeline, including ingestion, indexing, retrieval, generation, and evaluation.

You can run this in Google Colab or locally in VS Code.

In [ ]:
# Install dependencies if running in Colab
# !pip install -r requirements.txt

In [ ]:
import os
import sys

# Add the current directory to sys.path so we can import from src
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

## 1. Data Creation & Ingestion

In [ ]:
from src.ingestion import split_text

# Create dummy text data (simulating a PDF content)
dummy_text = """
RAG stands for Retrieval-Augmented Generation.
It combines an information retrieval component with a text generator model.
This allows the model to answer questions based on specific documents rather than just its training data.
Hugging Face provides many pre-trained models for both embedding and generation.
FAISS is a library for efficient similarity search and clustering of dense vectors.
"""

chunks = split_text(dummy_text, chunk_size=100, overlap=20)
print(f"Created {len(chunks)} chunks.")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

## 2. Indexing

In [ ]:
from src.indexing import Indexer

# Initialize Indexer
# We use a temporary index path for the demo
indexer = Indexer(index_path="demo_index.bin")

# Create vector store from chunks
indexer.create_vector_store(chunks)
indexer.save_index()
print("Index created and saved.")

## 3. Retrieval & Generation (RAG)

In [ ]:
from src.rag import RAGPipeline

# Initialize RAG Pipeline
rag = RAGPipeline(indexer)

query = "What is RAG?"
print(f"Query: {query}")

# 1. Retrieve
sources = rag.retrieve(query)
print("\nRetrieved Context:")
for s in sources:
    print(f"- {s[0]} (Score: {s[1]:.4f})")

# 2. Generate
answer = rag.query_rag(query)
print(f"\nAnswer:\n{answer}")

## 4. Evaluation

In [ ]:
from src.evaluation import calculate_recall, calculate_mrr, LLMJudge

# Retrieval Metrics
# Assume the first chunk is the relevant one
relevant_chunk = chunks[0]
retrieved_texts = [s[0] for s in sources]

recall = calculate_recall(retrieved_texts, [relevant_chunk], k=3)
print(f"Recall@3: {recall}")

# LLM Judge
print("\nRunning LLM Judge...")
judge = LLMJudge()
score = judge.evaluate_answer(
    question=query,
    answer=answer,
    context=sources[0][0] # Using top source as context context
)
print(f"Evaluation:\n{score}")

In [ ]:
# Cleanup
import os
if os.path.exists("demo_index.bin"):
    os.remove("demo_index.bin")
if os.path.exists("demo_index.bin.chunks"):
    os.remove("demo_index.bin.chunks")